# Riksdagens tjänstgörande personer: bildluckor i Wikidata

Källa: [Riksdagens personlista](https://data.riksdagen.se/personlista/?iid=&fnamn=&enamn=&f_ar=&kn=&parti=&valkrets=&org=).

Matcha `sourceid` → **P8388** och kontrollera **P18**. Kontrollera också `intressent_id` → **P1214** vid utebliven GUID-matchning. Namnsökning på exakta svenska/engelska etiketter ger möjliga kandidater; namnmatchning godkänns aldrig automatiskt.

Notebooken innehåller en inbyggd datasnapshot från **2026-10-07** och fungerar utan nätverk med `REFRESH = False`. Ändra till `True` för att hämta nya källdata. Riksdagens lista omfattar även tjänstgörande ersättare. Den är inte ett historiskt register över alla folkvalda.

**Utdata:** sökbar HTML-rapport, hela jämförelsen i CSV/JSON samt sammanfattning. Inga ändringar görs i Wikidata.

In [1]:
# Kör hela notebooken: Cell → Run All. Ändra REFRESH till True för aktuella API-data.
# Kräver Python 3 och requests: pip install requests
from pathlib import Path
import json, base64, zlib
REFRESH = False
OUTPUT_DIR = Path('riksdagen_audit_20261007')
OUTPUT_DIR.mkdir(exist_ok=True)
SNAPSHOT = '...'
for filename, data in json.loads(zlib.decompress(base64.b64decode(SNAPSHOT))).items():
    path = OUTPUT_DIR / filename
    if not path.exists():
        path.write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')
print('Utdata:', OUTPUT_DIR.resolve())


Utdata: /Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261007


## Hämtning, jämförelse och rapportfunktioner
SPARQL använder truthy-värden: föråldrade utsagor räknas inte och bästa rank styr vilka värden som syns. API-fel avbryter körningen. Wikidatas index kan släpa efter senaste ändringar.

In [2]:
"""Riksdagen → Wikidata: identifierarmatchning och bildkontroll. Python 3 + requests."""
import csv
import html
import json
from collections import defaultdict, Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlencode

import requests

SOURCE = 'https://data.riksdagen.se/personlista/?iid=&fnamn=&enamn=&f_ar=&kn=&parti=&valkrets=&org=&utformat=json'
ENDPOINT = 'https://query.wikidata.org/sparql'
Q8388 = 'SELECT ?item ?id ?image WHERE {?item wdt:P8388 ?id. OPTIONAL {?item wdt:P18 ?image}}'
Q1214 = 'SELECT ?item ?id WHERE { ?item wdt:P1214 ?id . }'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 RiksdagenImageAudit/1.0 (Salgo60)', 'Accept': 'application/json'})

def fetch(url, path, refresh=False, params=None):
    path = Path(path)
    if not refresh and path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    response = SESSION.get(url, params=params, timeout=90)
    response.raise_for_status()  # Fel får aldrig tolkas som saknade personer/bilder.
    data = response.json()
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    return data

def sparql(query, path, refresh=False):
    return fetch(ENDPOINT, path, refresh, {'query': query, 'format': 'json'})['results']['bindings']

def index(bindings):
    result = defaultdict(lambda: defaultdict(set))
    for b in bindings:
        qid = b['item']['value'].rsplit('/', 1)[-1]
        images = result[b['id']['value'].strip().lower()][qid]
        if 'image' in b:
            images.add(b['image']['value'])
    return result

def run(directory='.', refresh=False):
    out = Path(directory)
    out.mkdir(parents=True, exist_ok=True)
    raw = fetch(SOURCE, out/'riksdagen_personlista.json', refresh)
    people = raw['personlista']['person']
    if isinstance(people, dict):
        people = [people]
    assert people and len(people) == int(raw['personlista']['@hits'])
    assert len({p['sourceid'].lower() for p in people}) == len(people), 'Dubbletter i Riksdagens källa'
    guid_index = index(sparql(Q8388, out/'wikidata_p8388.json', refresh))
    old_index = index(sparql(Q1214, out/'wikidata_p1214.json', refresh))
    unmatched = [p for p in people if p['sourceid'].lower() not in guid_index]
    values = ' '.join(json.dumps(p['tilltalsnamn']+' '+p['efternamn'], ensure_ascii=False)+'@'+lang
                      for p in unmatched for lang in ('sv', 'en'))
    candidates = []
    if values:
        query = ('SELECT DISTINCT ?item ?label ?description ?birth ?image WHERE { VALUES ?label { '+values+
                 ' } ?item rdfs:label ?label . OPTIONAL { ?item schema:description ?description . FILTER(LANG(?description)="sv") }'
                 ' OPTIONAL { ?item wdt:P569 ?birth } OPTIONAL { ?item wdt:P18 ?image } }')
        candidates = sparql(query, out/'wikidata_name_candidates.json', refresh)
    candidate_index = defaultdict(dict)
    for c in candidates:
        qid = c['item']['value'].rsplit('/', 1)[-1]
        candidate_index[c['label']['value']][qid] = {
            'qid': qid, 'description': c.get('description', {}).get('value', ''),
            'birth': c.get('birth', {}).get('value', '')[:4]}
    rows = []
    for p in people:
        name = p['tilltalsnamn']+' '+p['efternamn']
        matches = guid_index.get(p['sourceid'].lower(), {})
        fallback = old_index.get(p['intressent_id'], {}) if not matches else {}
        fallback_images = set()
        for qid in fallback:
            # Direkt kontroll av P18 för personer hittade via det äldre ID:t.
            entity = fetch('https://www.wikidata.org/wiki/Special:EntityData/'+qid+'.json',
                           out/(qid+'_entity.json'), refresh)['entities'][qid]
            for claim in entity.get('claims', {}).get('P18', []):
                if claim.get('rank') != 'deprecated' and claim['mainsnak'].get('snaktype') == 'value':
                    fallback_images.add(claim['mainsnak']['datavalue']['value'])
        images = sorted({i for img in matches.values() for i in img})
        if len(matches) > 1:
            status = 'Flera P8388-matchningar'
        elif matches:
            status = 'Har bild (P18)' if images else 'Saknar bild (P18)'
        elif fallback:
            status = 'Hittad via P1214 – saknar P8388'
        else:
            status = 'Ingen identifierarmatchning – kontrollera Wikidata'
        rows.append({'namn': name, 'parti': p['parti'], 'valkrets': p['valkrets'],
                     'riksdagsstatus': p['status'], 'fodd_ar': p['fodd_ar'], 'sourceid': p['sourceid'],
                     'intressent_id': p['intressent_id'], 'qid': '; '.join(sorted(matches or fallback)),
                     'status': status, 'bilder': '; '.join(images),
                     'p1214_bildstatus': ('Har P18' if fallback_images else 'Saknar P18') if fallback else '',
                     'riksdagen_bild': p.get('bild_url_192', ''),
                     'kandidater': list(candidate_index[name].values()) if not matches else []})
    rows.sort(key=lambda r: (r['parti'], r['namn']))
    counts = Counter(r['status'] for r in rows)
    assert sum(counts.values()) == len(people)
    stamp = datetime.now(timezone.utc).isoformat()
    summary = {'rapport_genererad_utc': stamp, 'riksdagen_systemdatum': raw['personlista']['@systemdatum'],
               'antal_personer': len(rows), 'status': dict(counts), 'live_refresh': refresh}
    (out/'sammanfattning.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    (out/'bildkontroll_resultat.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    fields = [k for k in rows[0] if k != 'kandidater']
    with (out/'bildkontroll_resultat.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
    render_html(rows, summary, out/'riksdagen_saknar_bild.html')
    return rows, summary

def render_html(rows, summary, path):
    e = html.escape
    def link(url, label):
        return '<a target="_blank" rel="noopener" href="'+e(url, quote=True)+'">'+e(label)+'</a>'
    cards = []
    for r in rows:
        if r['status'] == 'Har bild (P18)':
            continue
        source = 'https://data.riksdagen.se/personlista/?'+urlencode({'iid': r['sourceid'], 'utformat': 'html'})
        wdlinks = ' · '.join(link('https://www.wikidata.org/wiki/'+qid, qid) for qid in r['qid'].split('; ') if qid)
        search = link('https://www.wikidata.org/w/index.php?'+urlencode({'search': r['namn'], 'title': 'Special:Search', 'ns0': '1'}), 'Sök i Wikidata')
        candidate_links = '<br>'.join(link('https://www.wikidata.org/wiki/'+c['qid'], c['qid'])+' '+
                                      e(c['description'])+' (född '+e(c['birth'] or '?')+')' for c in r['kandidater'])
        photo = ('<img loading="lazy" alt="Riksdagens porträtt av '+e(r['namn'], quote=True)+'" src="'+e(r['riksdagen_bild'], quote=True)+'">') if r['riksdagen_bild'] else ''
        cards.append('<article data-status="'+e(r['status'], quote=True)+'" data-party="'+e(r['parti'], quote=True)+'">'+photo+
                     '<div><span class="badge">'+e(r['status'])+'</span><h2>'+e(r['namn'])+' <small>'+e(r['parti'])+'</small></h2>'+
                     '<p>'+e(r['valkrets'])+' · Född '+e(r['fodd_ar'])+'<br>'+e(r['riksdagsstatus'])+'</p>'+
                     '<p>'+link(source, 'Riksdagen')+' · '+search+(' · '+wdlinks if wdlinks else '')+'</p>'+
                     '<p class="id">sourceid / P8388: '+e(r['sourceid'])+'<br>intressent_id / P1214: '+e(r['intressent_id'])+'</p>'+
                     ('<p>Bildkontroll av P1214-objekt: '+e(r['p1214_bildstatus'])+'</p>' if r['p1214_bildstatus'] else '')+
                     ('<details><summary>Namnkandidater – kräver identitetskontroll</summary>'+candidate_links+'</details>' if candidate_links else '')+'</div></article>')
    options = ''.join('<option>'+e(s)+'</option>' for s in summary['status'] if s != 'Har bild (P18)')
    parties = ''.join('<option>'+e(s)+'</option>' for s in sorted({r['parti'] for r in rows}))
    stats = ''.join('<div><strong>'+str(n)+'</strong><span>'+e(s)+'</span></div>' for s, n in summary['status'].items())
    document = '''<!doctype html><html lang="sv"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Riksdagen: bildluckor och Wikidata-matchning</title><style>
body{margin:0;background:#f3f6f9;color:#162538;font:16px/1.5 system-ui,sans-serif}main{max-width:1160px;margin:auto;padding:32px 24px}h1{font-size:34px;line-height:1.15}a{color:#075da6}small{font-size:16px;color:#536173}.stats{display:flex;flex-wrap:wrap;gap:12px;margin:24px 0}.stats div{background:#fff;padding:18px;border-radius:10px;flex:1;min-width:160px}.stats strong{display:block;font-size:32px}.stats span{font-size:14px}.note{background:#e6eef7;border-left:4px solid #376caa;padding:16px}nav{display:flex;flex-wrap:wrap;gap:10px;margin:24px 0}input,select{font:inherit;padding:10px;border:1px solid #aebbc9;border-radius:6px}input{flex:1;min-width:220px}#cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:16px}article{background:#fff;border:1px solid #dce3eb;border-radius:10px;padding:18px;display:flex;gap:18px}article[hidden]{display:none}img{width:96px;height:128px;object-fit:cover;border-radius:5px}h2{font-size:21px;margin:12px 0 4px}p{margin:9px 0}.badge{font-size:12px;background:#eef1f6;padding:4px 8px;border-radius:5px}.id{font-size:12px;overflow-wrap:anywhere;color:#536173}details{font-size:14px}footer{margin:32px 0;color:#536173;font-size:13px}@media(max-width:500px){#cards{grid-template-columns:1fr}img{width:64px;height:86px}main{padding:18px}article{gap:12px}}@media print{nav{display:none}article{break-inside:avoid}}
</style><main><h1>Riksdagen → Wikidata<br>Bildluckor och saknade kopplingar</h1>'''
    document += '<p>Källans systemdatum: <strong>'+e(summary['riksdagen_systemdatum'])+'</strong>. '+str(summary['antal_personer'])+' tjänstgörande personer i Riksdagens lista, inklusive ersättare. Rapport genererad '+e(summary['rapport_genererad_utc'])+'.</p>'
    document += '<section class="stats">'+stats+'</section><div class="note"><strong>Så läses resultatet</strong><br>sourceid matchas mot P8388. P18 kontrolleras i Wikidatas SPARQL-index (icke föråldrade truthy-värden). En utebliven identifierarmatchning bevisar inte att personen saknas i Wikidata. P1214 är en extra kontroll; namnkandidater med exakt svensk eller engelsk etikett kräver manuell kontroll av identitet, födelseår och politisk roll. Indexet kan släpa efter senaste ändringar.<br><br>Porträtten är länkade från Riksdagens bildarkiv och behöver internet. Rapporten fastställer ingen bildlicens och föreslår ingen automatisk uppladdning till Commons.</div>'
    document += '<nav><input id="search" placeholder="Sök namn, valkrets, Q-id eller identifierare" aria-label="Sök"><select id="status" aria-label="Status"><option value="">Alla problemtyper</option>'+options+'</select><select id="party" aria-label="Parti"><option value="">Alla partier</option>'+parties+'</select></nav><p id="visible"></p><section id="cards">'+''.join(cards)+'</section>'
    document += '<footer>Källor: '+link(SOURCE,'Riksdagens personlista')+' · '+link('https://www.wikidata.org/wiki/Property:P8388','P8388')+' · '+link('https://www.wikidata.org/wiki/Property:P1214','P1214')+' · '+link('https://www.wikidata.org/wiki/Property:P18','P18')+'. Personer med bild via P8388 visas endast i sammanfattningen; hela jämförelsen finns i CSV/JSON i paketet.</footer></main>'
    document += '''<script>const cards=[...document.querySelectorAll('article')];const search=document.getElementById('search'),status=document.getElementById('status'),party=document.getElementById('party');function filter(){let n=0;const q=search.value.toLocaleLowerCase('sv');for(const c of cards){const show=c.textContent.toLocaleLowerCase('sv').includes(q)&&(!status.value||c.dataset.status===status.value)&&(!party.value||c.dataset.party===party.value);c.hidden=!show;if(show)n++;}document.getElementById('visible').textContent=n+' av '+cards.length+' personer visas';}for(const x of [search,status,party])x.addEventListener('input',filter);filter();</script></html>'''
    path.write_text(document, encoding='utf-8')



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


In [3]:
rows, summary = run(OUTPUT_DIR, refresh=REFRESH)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "rapport_genererad_utc": "2026-10-07T12:38:35.197814+00:00",
  "riksdagen_systemdatum": "2026-10-07 13:13:28",
  "antal_personer": 349,
  "status": {
    "Saknar bild (P18)": 148,
    "Har bild (P18)": 155,
    "Ingen identifierarmatchning – kontrollera Wikidata": 45,
    "Hittad via P1214 – saknar P8388": 1
  },
  "live_refresh": false
}


In [4]:
from IPython.display import display, HTML, FileLink
display(FileLink(str(OUTPUT_DIR / 'riksdagen_saknar_bild.html')))
display(FileLink(str(OUTPUT_DIR / 'bildkontroll_resultat.csv')))
print('Omatchade med namnkandidater:', sum(bool(r['kandidater']) for r in rows if r['status'].startswith('Ingen')))

/Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261007/riksdagen_saknar_bild.html

/Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261007/bildkontroll_resultat.csv

Omatchade med namnkandidater: 38


## Tolkning

- **Saknar bild (P18):** matchad via P8388, inget truthy-bildvärde i Wikidata-indexet.
- **Hittad via P1214 – saknar P8388:** finns via ett annat stabilt person-ID; P18 kontrolleras direkt i objektets JSON.
- **Ingen identifierarmatchning:** kan saknas helt eller finnas utan de kontrollerade identifierarna. Granska födelseår, parti och politisk roll i namnkandidaterna före eventuell nyregistrering. Avsaknad av exakta namnträffar bevisar inte heller att objekt saknas.

Rapportens Riksdagsporträtt är externa bildlänkar. Rapporten anger ingen fastställd bildlicens. HTML-tabellen och filtren fungerar utan nätverk; porträtten kräver internet.

Snapshotfilerna återanvänds när `REFRESH=False`. Efter en uppdatering används de senast hämtade filerna i samma utdatakatalog. Det finns inga automatiska skrivoperationer mot Wikidata eller Commons.